## Semantic Chunks

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20243.35it/s]


In [18]:
text = """
LangChain can be used to connect language models with external tools and data sources.
A retriever in LangChain helps find relevant documents before generating an answer.
LangChain supports building RAG applications using vector databases and embeddings.
LangChain supports building Agentic & Generative AI applications using vector databases and embeddings.
Prompt templates make it easier to maintain consistent instructions across different LLM calls.
Steve Jobs and Steve Wozniak started Apple in the year 1976
Steve Jobs was the CEO of apple from 1997-2011
Apple is the greatest company to ever exist on the planet
Apple is the best startup on the palnet
Apple is the best company in the world
"""

In [19]:
list = text.split("\n")
list

['',
 'LangChain can be used to connect language models with external tools and data sources.',
 'A retriever in LangChain helps find relevant documents before generating an answer.',
 'LangChain supports building RAG applications using vector databases and embeddings.',
 'LangChain supports building Agentic & Generative AI applications using vector databases and embeddings.',
 'Prompt templates make it easier to maintain consistent instructions across different LLM calls.',
 'Steve Jobs and Steve Wozniak started Apple in the year 1976',
 'Steve Jobs was the CEO of apple from 1997-2011',
 'Apple is the greatest company to ever exist on the planet',
 'Apple is the best startup on the palnet',
 'Apple is the best company in the world',
 '']

In [20]:
## Step1: Split into sentences

sentences = [s.strip() for s in text.split("\n") if s.strip()]

## Step2: Embed each sentence

embeddings = model.encode_document(sentences)



In [21]:
embeddings

array([[-7.67038614e-02, -8.57952386e-02, -1.83175188e-02, ...,
         4.11281325e-02,  1.21268272e-01,  2.37027481e-02],
       [-8.44820812e-02,  5.83945252e-02,  6.64225227e-05, ...,
         6.97810724e-02,  1.11984268e-01,  4.32713218e-02],
       [-7.10447878e-02,  3.28512341e-02,  5.55018969e-02, ...,
        -5.92067093e-02,  7.41741806e-02,  2.05512810e-02],
       ...,
       [ 2.56531709e-03, -4.90415236e-03,  3.85434888e-02, ...,
        -8.33370760e-02,  1.52302414e-01,  4.26572561e-02],
       [ 1.52222775e-02, -5.81972301e-02,  9.17176902e-02, ...,
        -2.74978131e-02,  4.07203287e-02,  8.46477109e-04],
       [ 6.75330916e-03, -2.09666155e-02,  5.23997955e-02, ...,
        -7.22041503e-02,  1.35411754e-01,  6.82196394e-02]],
      shape=(10, 384), dtype=float32)

In [22]:
# Step3: Initialize parameters

threshold = 0.7 # control chunk tightness
chunks = []
current_chunks = [sentences[0]]

## Step4: Semantic Grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]

    if sim >= threshold:
        current_chunks.append(sentences[i])
    else:
        chunks.append(". ".join(current_chunks))
        current_chunks = [sentences[i]]

## Append the last chunk:

chunks.append(". ".join(current_chunks))

print("Semantic Chunks\n")

for i , chunk in enumerate(chunks):
    print(f"Chunk: {i+1}\n {chunk}")

Semantic Chunks

Chunk: 1
 LangChain can be used to connect language models with external tools and data sources.
Chunk: 2
 A retriever in LangChain helps find relevant documents before generating an answer.
Chunk: 3
 LangChain supports building RAG applications using vector databases and embeddings.
Chunk: 4
 LangChain supports building Agentic & Generative AI applications using vector databases and embeddings.
Chunk: 5
 Prompt templates make it easier to maintain consistent instructions across different LLM calls.
Chunk: 6
 Steve Jobs and Steve Wozniak started Apple in the year 1976. Steve Jobs was the CEO of apple from 1997-2011
Chunk: 7
 Apple is the greatest company to ever exist on the planet
Chunk: 8
 Apple is the best startup on the palnet
Chunk: 9
 Apple is the best company in the world


In [ ]:
chunks

['LangChain can be used to connect language models with external tools and data sources.',
 'A retriever in LangChain helps find relevant documents before generating an answer.',
 'LangChain supports building RAG applications using vector databases and embeddings.',
 'Prompt templates make it easier to maintain consistent instructions across different LLM calls.',
 'Steve Jobs and Steve Wozniak started Apple in the year 1976. Steve Jobs was the CEO of apple from 1997-2011',
 'Apple is the greatest company to ever exist on the planet']

## RAG Pipeline using modular coding

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

/home/jaywardhan/RAG_Udemy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_28584/1611302548.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [30]:
### Customized Semantic Chunker with threshold

class ThresholdSemanticChunker:
    def __init__(self,model_name = "all-MiniLM-L6-v2",threshold = 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text: str):

        sentences = [s.strip() for s in text.split("\n") if s.strip()]

        embeddings = self.model.encode_document(sentences)

        chunks = []
        current_chunks = [sentences[0]]

        for i in range(1,len(sentences)):
            sim = cosine_similarity(
                [embeddings[i-1]],
                [embeddings[i]]
            )[0][0]

            if sim >= self.threshold:
                current_chunks.append(sentences[i])

            else:
                chunks.append(". ".join(current_chunks))
                current_chunks = [sentences[i]]

        chunks.append(". ".join(current_chunks))
        return chunks

    def split_documents(self,docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content = chunk, metadata = doc.metadata))

        return result


In [31]:
text = """
LangChain can be used to connect language models with external tools and data sources.
A retriever in LangChain helps find relevant documents before generating an answer.
LangChain supports building RAG applications using vector databases and embeddings.
LangChain supports building Agentic & Generative AI applications using vector databases and embeddings.
Prompt templates make it easier to maintain consistent instructions across different LLM calls.
Steve Jobs and Steve Wozniak started Apple in the year 1976
Steve Jobs was the CEO of apple from 1997-2011
Apple is the greatest company to ever exist on the planet
Apple is the best startup on the palnet
Apple is the best company in the world
"""

doc = Document(page_content = text)

In [32]:
doc

Document(metadata={}, page_content='\nLangChain can be used to connect language models with external tools and data sources.\nA retriever in LangChain helps find relevant documents before generating an answer.\nLangChain supports building RAG applications using vector databases and embeddings.\nLangChain supports building Agentic & Generative AI applications using vector databases and embeddings.\nPrompt templates make it easier to maintain consistent instructions across different LLM calls.\nSteve Jobs and Steve Wozniak started Apple in the year 1976\nSteve Jobs was the CEO of apple from 1997-2011\nApple is the greatest company to ever exist on the planet\nApple is the best startup on the palnet\nApple is the best company in the world\n')

In [33]:
chunker = ThresholdSemanticChunker(threshold = 0.7)
chunks = chunker.split_documents([doc])
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18923.88it/s]


[Document(metadata={}, page_content='LangChain can be used to connect language models with external tools and data sources.'),
 Document(metadata={}, page_content='A retriever in LangChain helps find relevant documents before generating an answer.'),
 Document(metadata={}, page_content='LangChain supports building RAG applications using vector databases and embeddings.'),
 Document(metadata={}, page_content='LangChain supports building Agentic & Generative AI applications using vector databases and embeddings.'),
 Document(metadata={}, page_content='Prompt templates make it easier to maintain consistent instructions across different LLM calls.'),
 Document(metadata={}, page_content='Steve Jobs and Steve Wozniak started Apple in the year 1976. Steve Jobs was the CEO of apple from 1997-2011'),
 Document(metadata={}, page_content='Apple is the greatest company to ever exist on the planet'),
 Document(metadata={}, page_content='Apple is the best startup on the palnet'),
 Document(metadata=

In [ ]:
vectorstore = FAISS.from_documents(
    documents = chunks,
    embedding = OpenAIEmbeddings(model = "text-embedding-3-small", dimensions = 1536)
)

In [35]:
vectorstore.save_local("FAISS_DB")

In [36]:
print(f"Total vectrors stored in database are: {vectorstore.index.ntotal}")

Total vectrors stored in database are: 9


In [37]:
retriver = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":3}
)

In [38]:
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [39]:
##LLM

llm = init_chat_model("openai:gpt-4o-mini", temperature = 0.2)

In [40]:
## LCEL Chain with retrieval

lcel_rag_chain = (
   RunnableMap(
       {
           "context": lambda x: retriver.invoke(x["question"]),
           "question": lambda x: x["question"]
       }
   )
   |prompt
   |llm
   |StrOutputParser()
)

In [41]:
query = {"question": "When was apple founded and who was it's CEO?"}
result = lcel_rag_chain.invoke(query)

result

'Apple was founded in the year 1976 by Steve Jobs and Steve Wozniak. Steve Jobs was the CEO of Apple from 1997 to 2011.'

## Semantic Chnuking With Langchain

In [42]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [47]:
loader = TextLoader(
    "/home/jaywardhan/RAG_Udemy/semantic_chunks_intro.txt"
)

docs = loader.load()
docs

[Document(metadata={'source': '/home/jaywardhan/RAG_Udemy/semantic_chunks_intro.txt'}, page_content='LangChain can be used to connect language models with external tools and data sources.\nA retriever in LangChain helps find relevant documents before generating an answer.\nLangChain supports building RAG applications using vector databases and embeddings.\nPrompt templates make it easier to maintain consistent instructions across different LLM calls.\nSteve Jobs and Steve Wozniak started Apple in the year 1976\nSteve Jobs was the CEO of apple from 1997-2011\nApple is the greatest company to ever exist on the planet')]

In [48]:
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    dimensions = 1536
)

In [49]:
## Semantic Chunker

chunker = SemanticChunker(embeddings)

## Split documents:
chunks = chunker.split_documents(docs)


In [50]:
for i , chunk in enumerate(chunks):
    print(f"Chunk: {i+1}\n")
    print(f"Content: {chunk.page_content}\n")
    print(f"Metadata: {chunk.metadata}\n")


Chunk: 1

Content: LangChain can be used to connect language models with external tools and data sources. A retriever in LangChain helps find relevant documents before generating an answer. LangChain supports building RAG applications using vector databases and embeddings.

Metadata: {'source': '/home/jaywardhan/RAG_Udemy/semantic_chunks_intro.txt'}

Chunk: 2

Content: Prompt templates make it easier to maintain consistent instructions across different LLM calls. Steve Jobs and Steve Wozniak started Apple in the year 1976
Steve Jobs was the CEO of apple from 1997-2011
Apple is the greatest company to ever exist on the planet

Metadata: {'source': '/home/jaywardhan/RAG_Udemy/semantic_chunks_intro.txt'}

